# 静息态 EEG 频域分析 - 单被试

**静息态数据特点**：
- 无时间锁定事件，不适合做时频分析 (TFR)
- 关注频域特征的稳定性
- 重点分析：PSD、SNR、频段功率、相对功率

In [1]:
# 导入必要的库
import os
import os.path as op
import mne
import matplotlib.pyplot as plt
import numpy as np
import pickle
import  numpy as np


# 设置中文字体（消除中文显示警告）
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

In [4]:
# 设置路径和被试列表
sub_ids = ['001', '021', '022', '024', '047', '049', '051', '071', '072', '073', '076', '092']
data_path = r'..\..\preprocessing\data\6epoch_clean\\'
save_path = r'.\results\single_subject\\'
fig_path = r'.\figures\single_subject\\'

# 创建输出目录
os.makedirs(save_path, exist_ok=True)
os.makedirs(fig_path, exist_ok=True)

# 选择测试被试
sub_id = sub_ids[3]
print(f'分析被试: {sub_id}')

分析被试: 024


## 分析方法

### 1. 功率谱密度 (PSD)
- **方法**: Welch方法
- **频率范围**: 1-40 Hz
- **窗口**: 10秒 (整个epoch)
- **频率分辨率**: 0.1 Hz

### 2. 信噪比 (SNR)
- 基于PSD计算
- 使用邻近频率作为噪声估计

### 3. 频段功率
- Delta: 1-4 Hz
- Theta: 4-8 Hz
- Alpha: 8-13 Hz
- Beta: 13-30 Hz
- Gamma: 30-40 Hz

### 4. 相对功率
- 各频段功率占总功率的百分比

In [14]:
def snr_spectrum(psd, noise_n_neighbor_freqs=1, noise_skip_neighbor_freqs=1):
    """计算SNR谱"""  
    averaging_kernel = np.concatenate((
        np.ones(noise_n_neighbor_freqs),
        np.zeros(2 * noise_skip_neighbor_freqs + 1),
        np.ones(noise_n_neighbor_freqs)))
    averaging_kernel /= averaging_kernel.sum()
    
    mean_noise = np.apply_along_axis(
        lambda psd_: np.convolve(psd_, averaging_kernel, mode="valid"),
        axis=-1, arr=psd
    )
    
    edge_width = noise_n_neighbor_freqs + noise_skip_neighbor_freqs
    pad_width = [(0, 0)] * (mean_noise.ndim - 1) + [(edge_width, edge_width)]
    mean_noise = np.pad(mean_noise, pad_width=pad_width, constant_values=np.nan)
    
    return psd / mean_noise

In [15]:
# 读取清理后的epochs数据
fname = op.join(data_path + sub_id + "-epo.fif")
epochs = mne.read_epochs(fname, preload=True, verbose=None)

print(f"被试 {sub_id}:")
print(f"  Epochs数量: {len(epochs)}")
print(f"  通道数: {len(epochs.ch_names)}")
print(f"  采样率: {epochs.info['sfreq']} Hz")
print(f"  时间范围: {epochs.tmin} 到 {epochs.tmax} 秒")

# 检查采样率，如果不是500Hz则重采样
TARGET_SFREQ = 500
if epochs.info['sfreq'] != TARGET_SFREQ:
    print(f"\n⚠ 采样率不是 {TARGET_SFREQ} Hz，正在重采样...")
    epochs = epochs.resample(TARGET_SFREQ)
    print(f"✓ 已重采样到 {TARGET_SFREQ} Hz")
    print(f"  新采样点数: {len(epochs.times)}")
else:
    print(f"\n✓ 采样率正确: {TARGET_SFREQ} Hz")

Reading d:\LYW\REST_COM\analysis\频域分析\..\..\preprocessing\data\6epoch_clean\024-epo.fif ...
    Found the data of interest:
        t =       0.00 ...   10000.00 ms
        0 CTF compensation matrices available
Not setting metadata
25 matching events found
No baseline correction applied
0 projection items activated
被试 024:
  Epochs数量: 25
  通道数: 60
  采样率: 500.0 Hz
  时间范围: 0.0 到 10.0 秒

✓ 采样率正确: 500 Hz


## 步骤 1: 功率谱密度 (PSD) 分析

In [16]:
# PSD参数设置
tmin = 0.
tmax = None  # 使用整个epoch长度
fmin = 1.
fmax = 40.

# 获取实际的epoch信息
sfreq = epochs.info['sfreq']
n_times = epochs.get_data().shape[2]
epoch_duration = n_times / sfreq

print(f"采样率: {sfreq} Hz")
print(f"Epoch样本数: {n_times}")
print(f"Epoch时长: {epoch_duration:.2f} 秒")

# 设置n_fft（不超过样本数）
# 500 Hz × 10秒 = 5000 采样点，使用全部数据以获得最高频率分辨率
n_fft = n_times  # 使用全部样本数
print(f"n_fft: {n_fft}")
print(f"理论频率分辨率: {sfreq / n_fft:.3f} Hz")

# 计算PSD (Welch方法) - 使用新版MNE API
psd = epochs.compute_psd(
    method='welch',
    fmin=fmin, 
    fmax=fmax,
    tmin=tmin, 
    tmax=tmax,
    n_fft=n_fft,
    n_overlap=0,
    verbose=False
)

# 获取PSD数据和频率
psds_all = psd.get_data()  # shape: (n_epochs, n_channels, n_freqs)
freqs = psd.freqs

# 跨epochs平均
psds = psds_all.mean(axis=0)  # shape: (n_channels, n_freqs)

print(f"\nPSD原始 shape: {psds_all.shape}")
print(f"PSD平均后 shape: {psds.shape}")
print(f"频率点数: {len(freqs)}")
print(f"频率范围: {freqs[0]:.1f} - {freqs[-1]:.1f} Hz")
print(f"实际频率分辨率: {freqs[1] - freqs[0]:.3f} Hz")

采样率: 500.0 Hz
Epoch样本数: 5001
Epoch时长: 10.00 秒
n_fft: 5001
理论频率分辨率: 0.100 Hz

PSD原始 shape: (25, 59, 390)
PSD平均后 shape: (59, 390)
频率点数: 390
频率范围: 1.1 - 40.0 Hz
实际频率分辨率: 0.100 Hz


In [17]:
# 可视化PSD
%matplotlib qt

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(freqs, psds.T, alpha=0.3, color="gray")
ax.plot(freqs, psds.mean(axis=0), color="red", linewidth=2, label="平均")
ax.set_ylabel("功率谱密度 (μV²/Hz)")
ax.set_title(f"被试 {sub_id} - 功率谱密度")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f"{fig_path}{sub_id}_psd.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"✓ 已保存: {fig_path}{sub_id}_psd.png")

C:\Users\Administrator\AppData\Local\Temp\ipykernel_232980\1744959383.py:11: UserWarning: Glyph 178 (\N{SUPERSCRIPT TWO}) missing from current font.
  plt.tight_layout()
C:\Users\Administrator\AppData\Local\Temp\ipykernel_232980\1744959383.py:12: UserWarning: Glyph 178 (\N{SUPERSCRIPT TWO}) missing from current font.
  plt.savefig(f"{fig_path}{sub_id}_psd.png", dpi=150, bbox_inches="tight")


✓ 已保存: .\figures\single_subject\\024_psd.png


## 步骤 2: 信噪比 (SNR) 分析

In [18]:
# 计算SNR
snrs = snr_spectrum(psds, noise_n_neighbor_freqs=3, noise_skip_neighbor_freqs=0)

# 选择感兴趣的频率范围
freq_range = range(np.where(np.floor(freqs) == 1.)[0][0],
                   np.where(np.ceil(freqs) == fmax - 1)[0][0])

snr_mean = snrs.mean(axis=0)[freq_range]
snr_std = snrs.std(axis=0)[freq_range]
freqs_snr = freqs[freq_range]

print(f"SNR shape: {snrs.shape}")
print(f"SNR频率范围: {freqs_snr[0]:.1f} - {freqs_snr[-1]:.1f} Hz")

SNR shape: (59, 390)
SNR频率范围: 1.1 - 38.0 Hz


d:\ProgramData\anaconda3\envs\mne12_ADHDproject\lib\site-packages\ipykernel\eventloops.py:128: UserWarning: Glyph 178 (\N{SUPERSCRIPT TWO}) missing from current font.
  el.exec() if hasattr(el, 'exec') else el.exec_()


In [19]:
# 可视化SNR
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(freqs_snr, snr_mean, color="blue", linewidth=2)
ax.fill_between(freqs_snr, snr_mean - snr_std, snr_mean + snr_std, alpha=0.3, color="blue")
ax.axhline(y=1, color="red", linestyle="--", label="SNR=1")
ax.set_xlabel("频率 (Hz)")
ax.set_ylabel("信噪比 (SNR)")
ax.set_title(f"被试 {sub_id} - 信噪比谱")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f"{fig_path}{sub_id}_snr.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"✓ 已保存: {fig_path}{sub_id}_snr.png")

✓ 已保存: .\figures\single_subject\\024_snr.png


## 步骤 3: 频段功率分析

In [20]:
# 定义频段
bands = {
    "Delta": (1, 4),
    "Theta": (4, 8),
    "Alpha": (8, 13),
    "Beta": (13, 30),
    "Gamma": (30, 40)
}

# 计算每个频段的平均功率
band_powers = {}
for band_name, (fmin_band, fmax_band) in bands.items():
    freq_idx = np.where((freqs >= fmin_band) & (freqs <= fmax_band))[0]
    band_power = psds[:, freq_idx].mean(axis=1)
    band_powers[band_name] = band_power
    print(f"{band_name} ({fmin_band}-{fmax_band} Hz): {band_power.mean():.2e} μV²/Hz")

print("频段功率计算完成")

Delta (1-4 Hz): 1.56e-11 μV²/Hz
Theta (4-8 Hz): 5.20e-12 μV²/Hz
Alpha (8-13 Hz): 1.21e-12 μV²/Hz
Beta (13-30 Hz): 4.63e-13 μV²/Hz
Gamma (30-40 Hz): 6.99e-14 μV²/Hz
频段功率计算完成


In [21]:
# 可视化频段功率拓扑图
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, (band_name, band_power) in enumerate(band_powers.items()):
    if idx < len(axes):
        im, _ = mne.viz.plot_topomap(
            band_power, epochs.info, axes=axes[idx],
            show=False, cmap="RdBu_r"
        )
        axes[idx].set_title(f"{band_name} ({bands[band_name][0]}-{bands[band_name][1]} Hz)")
        plt.colorbar(im, ax=axes[idx], fraction=0.046, pad=0.04)

axes[-1].axis("off")
plt.suptitle(f"被试 {sub_id} - 频段功率拓扑图", fontsize=14, y=0.98)
plt.tight_layout()
plt.savefig(f"{fig_path}{sub_id}_band_powers.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"✓ 已保存: {fig_path}{sub_id}_band_powers.png")

✓ 已保存: .\figures\single_subject\\024_band_powers.png


## 步骤 4: 相对功率分析

计算各频段功率占总功率的百分比

In [22]:
# 计算总功率
total_power = sum(band_powers.values())

# 计算相对功率
relative_powers = {}
for band_name, band_power in band_powers.items():
    relative_power = (band_power / total_power) * 100
    relative_powers[band_name] = relative_power
    print(f"{band_name}: {relative_power.mean():.2f}%")

print("相对功率计算完成")

Delta: 68.52%
Theta: 23.99%
Alpha: 5.35%
Beta: 1.85%
Gamma: 0.28%
相对功率计算完成


## 步骤 5: 保存分析结果

In [23]:
# 保存所有分析结果
results = {
    "subject_id": sub_id,
    "psd": {"psds": psds, "freqs": freqs},
    "snr": {"snrs": snrs, "snr_mean": snr_mean, "snr_std": snr_std, "freqs_snr": freqs_snr},
    "band_powers": band_powers,
    "relative_powers": relative_powers,
    "bands": bands
}

result_file = f"{save_path}{sub_id}_frequency_results.pkl"
with open(result_file, "wb") as f:
    pickle.dump(results, f)

print(f"✓ 已保存分析结果: {result_file}")
print(f"结果包含:")
print(f"  - PSD: {psds.shape}")
print(f"  - SNR: {snrs.shape}")
print(f"  - 频段功率: {len(band_powers)} 个频段")
print(f"  - 相对功率: {len(relative_powers)} 个频段")

✓ 已保存分析结果: .\results\single_subject\\024_frequency_results.pkl
结果包含:
  - PSD: (59, 390)
  - SNR: (59, 390)
  - 频段功率: 5 个频段
  - 相对功率: 5 个频段
